<a href="https://colab.research.google.com/github/anu5hkaa/AI-War-News-Analyser-System/blob/main/War_news_analzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
import numpy as np

In [2]:
api_key='e2a5d3214eeb40e297e0ab4e64e4b510'

In [3]:

url = "https://newsapi.org/v2/everything"

war_keywords = [
    "war", "conflict", "military", "airstrike", "missile",
    "bomb", "attack", "invasion", "army", "troops",
    "defense", "clash", "border", "violence", "strike"
]
queries = [
    "war",
    "military conflict",
    "airstrike attack",
    "border clash",
    "missile strike",
    "army operation",
    "defense military",
    "troops conflict"
]

all_articles = []
news_list = []
for query in queries:
    for page in range(1, 6):   # 5 pages × 20 = 100 per query

        params = {
            "q": query,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 20,   # free API limit per request
            "page": page,
            "apiKey": api_key
        }

        response = requests.get(url, params=params)
        data = response.json()

        articles = data.get("articles", [])
        all_articles.extend(articles)

for article in all_articles:
    news = {
        "title": article.get("title", ""),
        "content": article.get("description", ""),
        "source": article.get("source", {}).get("name", ""),
        "url": article.get("url", ""),
        "date": article.get("publishedAt", "")
    }
    news_list.append(news)

In [4]:

df = pd.DataFrame(news_list)

In [5]:
df.head(10)

,title,content,source,url,date
0,A 40-year-old Iran tariff quietly built Americ...,"Benefiting off viral trends, like Dubai chocol...",Salon,https://www.salon.com/2026/05/06/a-40-year-old...,2026-05-06T10:30:26Z
1,This is what super powers must learn from US’ ...,"From the Great Game to the Cold War, periphera...",RT,https://www.rt.com/news/639497-what-super-powe...,2026-05-06T10:30:06Z
2,UK services industry faces 'short-lived' rebound,Growth in the UK's services sector rebounded l...,RTE,https://www.rte.ie/news/business/2026/0506/157...,2026-05-06T10:30:02Z
3,Perrigo Reports First Quarter 2026 Financial R...,Mitigating category headwinds with market shar...,PRNewswire,https://www.prnewswire.com/news-releases/perri...,2026-05-06T10:30:00Z
4,Canadian lettuce grower challenges American mo...,Most of the lettuce Canadians eat travels thou...,Financial Post,https://financialpost.com/feature/canadian-let...,2026-05-06T10:27:30Z
5,Geopolitics Strategic Intelligence Executive B...,The Middle East conflict underscores geopoliti...,GlobeNewswire,https://www.globenewswire.com/news-release/202...,2026-05-06T10:26:00Z
6,"Bitcoin moves above $82,000 while ZEC and DASH...","BTC climbed above $82,000 as a weaker dollar l...",CoinDesk,https://www.coindesk.com/markets/2026/05/06/bi...,2026-05-06T10:25:43Z
7,Egypt Opens High-Speed Monorail Linking Cairo ...,Egypt opened the first phase of a showpiece mo...,Financial Post,https://financialpost.com/pmn/business-pmn/egy...,2026-05-06T10:25:16Z
8,Saudi Venture Firms Press Ahead With Fundraisi...,Saudi Arabian venture capital firms are pressi...,Financial Post,https://financialpost.com/pmn/business-pmn/sau...,2026-05-06T10:25:02Z
9,Mubi Takes Lukas Dhont's 'Coward' in Multiple ...,Dhont's follow-up to his Oscar-nominated 'Clos...,Hollywood Reporter,http://www.hollywoodreporter.com/movies/movie-...,2026-05-06T10:25:00Z


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 374 entries, 0 to 373
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    374 non-null    object
 1   content  370 non-null    object
 2   source   374 non-null    object
 3   url      374 non-null    object
 4   date     374 non-null    object
dtypes: object(5)
memory usage: 14.7+ KB


In [7]:
df['content'][4]

'Most of the lettuce Canadians eat travels thousands of kilometres from California to our plates. Jay Willmot wants to change that. Read more'

In [8]:
len((df))

374

In [9]:
df = df[~df["title"].str.lower().str.contains("movie|film|trailer|review|box office")]

In [10]:
len(df)

369

In [11]:
from transformers import pipeline

# Load once (don’t put inside loop)
classifier = pipeline("zero-shot-classification")

def is_war_news(text):
    labels = ["war or military news", "not related"]

    result = classifier(text, labels)

    return result["labels"][0] == "war or military news"

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [12]:
df = df[df["title"].apply(is_war_news)]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [13]:
print(len(df))


284


In [14]:
!pip install sentence-transformers faiss-cpu

In [15]:
from sentence_transformers import SentenceTransformer

# Load the SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')  # This is a good, efficient model

df["title"] = df["title"].fillna('').str.lower()
df["content"] = df["content"].fillna('').str.lower()
df["source"] = df["source"].astype(str).str.lower()
# Prepare your text data: combine title and content or just use one
df['combined_text'] = df['title'] + " " + df['content']

# Convert the combined text into embeddings
embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [16]:
embeddings.shape


(284, 384)

In [17]:
!pip install faiss-cpu

In [18]:
import faiss
import numpy as np

embeddings = np.array(embeddings)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

In [19]:
query = "pakistan and india at war"

query_embedding = model.encode([query])
query_embedding = np.array(query_embedding)

faiss.normalize_L2(query_embedding)

In [20]:
D, I = index.search(query_embedding, k=5)

In [21]:
for i in I[0]:
    print(df.iloc[i]["title"])
    print(df.iloc[i]["source"])
    print(df.iloc[i]['url'])
    print("-----")

pakistan forces kill 22 militants in northwest clash, child killed near afghan border
the times of india
https://economictimes.indiatimes.com/news/defence/pakistan-forces-kill-22-militants-in-northwest-clash-child-killed-near-afghan-border/articleshow/130489247.cms
-----
afghan officials say pakistani strikes killed 7, wounded 85 in first attacks since peace talks
the times of india
https://economictimes.indiatimes.com/news/defence/afghan-officials-say-pakistani-strikes-killed-7-wounded-85-in-first-attacks-since-peace-talks/articleshow/130567138.cms
-----
ceasefire at risk as pakistan and afghanistan report cross-border attacks
al jazeera english
https://www.aljazeera.com/news/2026/4/27/ceasefire-at-risk-as-pakistan-and-afghanistan-report-cross-border-attacks
-----
afghan officials say pakistani strikes killed 7 and wounded 85 in first attacks since peace talks
japan today
https://japantoday.com/category/world/afghan-officials-say-pakistani-strikes-killed-7-and-wounded-85-in-first-atta

In [22]:
!pip install transformers

In [23]:
!pip install feedparser

In [24]:
import numpy as np
import pandas as pd
import faiss
import feedparser
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder


model = SentenceTransformer('all-MiniLM-L6-v2')
cross_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
df['content'] = df['content'].fillna('').astype(str).str.lower()
df['title'] = df['title'].fillna('').astype(str).str.lower()
df['date'] = df['date'].astype(str)

df['full_text'] = df['title'] + " " + df['content']

embeddings = model.encode(df['full_text'].tolist(), show_progress_bar=True)
embeddings = np.array(embeddings)
faiss.normalize_L2(embeddings)

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)


war_keywords = [
    "war", "conflict", "attack", "military", "missile",
    "battle", "army", "troops", "airstrike", "bomb",
    "border", "clash", "violence", "defense", "ceasefire"
]
def expand_query(query):
    return query + " war conflict military attack"


def smart_filter(results, query):
    words = [w for w in query.split() if len(w) > 2]

    filtered = []

    for item in results:
        text = (item["title"] + " " + item["content"]).lower()

        # must be war-related
        if not any(k in text for k in war_keywords):
            continue

        if len(words) == 1:
            # single word → flexible
            if words[0] in text:
                filtered.append(item)
        else:
            # multiple words → strict (ALL must match)
            if all(word in text for word in words):
                filtered.append(item)

    return filtered


def clean_google_link(link):
    try:
        response = requests.get(link, allow_redirects=True, timeout=5)
        return response.url
    except:
        return link

def fetch_rss_news(query):

    query_formatted = query.replace(" ", "+")

    rss_urls = [
        f"https://news.google.com/rss/search?q={query_formatted}+war",
        f"https://news.google.com/rss/search?q={query_formatted}+conflict",
        f"https://news.google.com/rss/search?q={query_formatted}+military",

        "https://www.defensenews.com/arc/outboundfeeds/rss/?outputType=xml",
        "https://www.militarytimes.com/arc/outboundfeeds/rss/",
        "https://www.armytimes.com/arc/outboundfeeds/rss/"
    ]

    rss_data = []

    for url in rss_urls:
        feed = feedparser.parse(url)

        for entry in feed.entries:
            text = entry.title.lower()

            if any(k in text for k in war_keywords):
                rss_data.append({
                    "title": entry.title.lower(),
                    "content": entry.title,
                    "url": clean_google_link(entry.link),
                    "date": entry.published if "published" in entry else "",
                    "full_text": entry.title.lower()
                })

    seen = set()
    unique = []

    for item in rss_data:
        if item["url"] not in seen:
            unique.append(item)
            seen.add(item["url"])

    return unique[:40]

def get_top_matches(query):

    original_query = query.lower().strip()
    expanded_query = expand_query(original_query)

    print("Searching...")


    query_embedding = model.encode([expanded_query])
    query_embedding = np.array(query_embedding)
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, k=20)

    api_results = []

    for i in I[0]:
        if i < len(df):
            api_results.append({
                "title": str(df.iloc[i]['title']),
                "content": str(df.iloc[i]['content'])[:1000],
                "url": df.iloc[i]['url'],
                "date": df.iloc[i]['date']
            })


    rss_results = fetch_rss_news(original_query)

    api_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in api_results]
    api_scores = cross_model.predict(api_pairs) if api_pairs else []
    api_ranked = sorted(zip(api_scores, api_results), reverse=True)

    rss_pairs = [(expanded_query, item["title"] + " " + item["content"]) for item in rss_results]
    rss_scores = cross_model.predict(rss_pairs) if rss_pairs else []
    rss_ranked = sorted(zip(rss_scores, rss_results), reverse=True)

    combined = [item for _, item in (api_ranked[:5] + rss_ranked[:5])]

    filtered = smart_filter(combined, original_query)

    if not filtered:
        print("Not Found")
        return []

    final_results = filtered[:10]

    clean_results = []

    for item in final_results:
        try:
            parsed = pd.to_datetime(item["date"], errors='coerce')
            date_str = parsed.strftime("%Y-%m-%d") if not pd.isna(parsed) else str(item["date"])
        except:
            date_str = str(item["date"])

        clean_results.append({
            "title": item["title"],
            "content": item["content"],
            "url": item["url"],
            "date": date_str
        })

    if len(clean_results) >= 2:
        prompt = create_prompt(original_query, clean_results)
        summary = get_llm_response(prompt)

        print("\nSummary:\n")
        print(summary)

    print("\nSources:\n")

    for i, news in enumerate(clean_results):
        print(f"{i+1}. {news['title']}")
        print(news["url"])
        print("Date:", news["date"])
        print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [25]:
!pip install Groq

In [32]:
import os
from getpass import getpass
os.environ["GROQ_API_KEY"]=getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [34]:
from groq import Groq
client=Groq(api_key=os.getenv("GROQ_API_KEY"))

In [35]:
def create_prompt(query, clean_results):

    prompt = f"""
You are a war news analyst.

User Query: {query}

Below are some news articles:

"""

    for i, news in enumerate(clean_results, 1):
        prompt += f"""
Article {i}:
Title: {news['title']}
Content: {news['content']}
Date: {news['date']}
"""

    prompt += """
Task:
- Summarize what is happening
- Explain in simple simple words
-Key insights:
- Keep answer short
"""

    return prompt

In [36]:
def get_llm_response(prompt):

    print("LLM running...")

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    print("LLM done!")

    return response.choices[0].message.content

In [48]:
get_top_matches("iran and us")


Searching...
LLM running...
LLM done!

Summary:

**What’s happening**

- A war that began in late February after a U.S.–Israeli strike on Iranian targets is now in its third month.  
- A fragile **temporary cease‑fire** is holding on the ground, but **naval clashes and retaliatory strikes** continue, especially around the Strait of Hormuz.  
- President **Donald Trump** (still in office in this timeline) is the public face of the U.S. effort. He has been alternating between **talking about negotiations** and **threatening more bombing** to push Iran back.  
- Iran’s **Islamic Revolutionary Guard (IRGC) Qods Force** has signaled that it could launch attacks against the U.S. mainland, but so far those plans have not materialised.  
- Both sides are now **working on a short‑term deal** to stop the fighting while they keep pressure on each other.

**Simple explanation**

- Fighting started after the U.S. and Israel hit Iran.  
- The armies have stopped most land battles for now, but ships 